# Heavy atoms and mass-weighted modes

> **New to mdCIF?** Run [1_basic](../1_basic/1_basic.ipynb) first. It explains each step (PCA,
> clustering, medoids, modes) and what ends up in the file. This notebook changes two settings.

| | 1_basic | here |
|---|---|---|
| atoms analysed | Cα only (56) | every C, N and O atom of the protein (434): backbone and side chains |
| each state's modes | PCA | mass-weighted PCA (quasi-harmonic analysis) |

**Heavy atoms** let the modes describe the side chains as well as the backbone. **Mass
weighting** makes each atom count in proportion to its mass, as in a vibrational analysis, so
the modes approximate the protein's slowest, lowest-frequency motions. With Cα atoms alone it
changes nothing (all the atoms weigh the same), so it goes with a selection of mixed atoms.

The map that the clustering uses (the global PCA) always stays plain, unweighted PCA, so that
each frame has a well-defined position on it.

The notebook writes `gb1_heavy_mw.cif` into this folder and runs in under a minute.

In [1]:
import os
import sys

os.environ.setdefault("OMP_NUM_THREADS", "4")  # be polite to shared nodes (set before numpy)
# Use only this environment's packages, even if a module system (e.g. Amber's) put another
# Python's site-packages on PYTHONPATH.
sys.path = [p for p in sys.path if "site-packages" not in p or p.startswith(sys.prefix)]

In [2]:
import warnings
from pathlib import Path

import numpy as np

from mdcif.pipeline import cluster, load_trajectory, save_mdcif
from mdcif.provenance import load_provenance
from mdcif.reader import read_mdcif
from mdcif.validate import validate_mdcif

warnings.filterwarnings("ignore", category=DeprecationWarning)  # MDAnalysis reader notices

DATA = Path("../../data/3gb1/meld")      # the GB1 MELD run of 1_basic, not in git
TOP = DATA / "3gb1_HMR.prmtop"           # Amber topology
TRAJ = DATA / "meld.nc.001"              # replica 1: 300 K
HEAVY = "protein and element C N O"      # MDAnalysis selection: all C, N and O atoms

traj = load_trajectory(TOP, TRAJ, selection=HEAVY, start=25_000, step=10)
elements, counts = np.unique(traj.analysis.elements, return_counts=True)
print(f"{traj.n_frames} frames, {len(traj.analysis)} atoms: "
      + ", ".join(f"{n} {e}" for e, n in zip(elements, counts)))

17683 frames, 434 atoms: 273 C, 67 N, 94 O


`selection=` sets the atoms for every step: superposition, the PCA map, medoids, precision and
modes. The file still holds every heavy atom of each model (in GB1 that adds the sulfur of
Met1, which `protein and not element H` would also analyse).

## 1. States

The clustering call is the same as in 1_basic: HDBSCAN on PCs 1–3, here the PCs of the
heavy-atom motion.

In [3]:
clusters = cluster(traj)
print(clusters.details)
for c, f in enumerate(clusters.representatives):
    print(f"model {c + 1}: {clusters.populations[c]:6d} frames "
          f"({100 * clusters.populations[c] / traj.n_frames:4.1f}%), "
          f"medoid {traj.frame_label(f)}, precision {clusters.precision[c]:.2f} Å")
print(f"noise: {100 * clusters.noise_fraction:.1f}% of frames")

HDBSCAN on PCA projection (global PCs 1-3), min_cluster_size 177, min_samples 177, cluster_selection eom
model 1:  15303 frames (86.5%), medoid frame 169490, t = 338.982 ns, precision 2.12 Å
model 2:    836 frames ( 4.7%), medoid frame 119750, t = 239.502 ns, precision 1.68 Å
model 3:    203 frames ( 1.1%), medoid frame 33450, t = 66.902 ns, precision 1.56 Å
model 4:    182 frames ( 1.0%), medoid frame 194380, t = 388.762 ns, precision 1.62 Å
noise: 6.6% of frames


The side chains change the map a little: besides the folded state (model 1) and the misfolded
state (model 2) of 1_basic, HDBSCAN now separates two small states of about 1% each. The
precision is now the mean heavy-atom RMSD to the medoid.

## 2. Mass-weighted modes

`mass_weighted=True` makes each state's modes quasi-harmonic. The masses are the standard ones
of each element (C 12.011, N 14.007, O 15.999), not the topology's: this run uses hydrogen mass
repartitioning, which moves mass from the heavy atoms to their hydrogens. The `modes:` block
of [`provenance.yaml`](provenance.yaml) also says `mass_weighted: true`, so `save_mdcif` would
take it from there as well.

In [4]:
summary = save_mdcif("gb1_heavy_mw.cif", traj, clusters, load_provenance("provenance.yaml"),
                     mass_weighted=True, reference="3GB1")
print(summary.report())
validate_mdcif("gb1_heavy_mw.cif")   # raises with the list of problems if anything is wrong
print("valid")

gb1_heavy_mw.cif: 17683 frames, 434 analysis atoms, 435 deposited atoms
  global PCA var% 49/11/7 (cum 66%)
  clustering: HDBSCAN on PCA projection (global PCs 1-3), min_cluster_size 177, min_samples 177, cluster_selection eom, 4 clusters, noise 6.6%
  model 1: 15303 frames (86.5%), medoid frame 169490, t = 338.982 ns, precision 2.12 A, RMSD to PDB 3GB1 model 1 2.78 A, modes var% 21/10/6 (cum 38%)
  model 2:   836 frames ( 4.7%), medoid frame 119750, t = 239.502 ns, precision 1.68 A, RMSD to PDB 3GB1 model 1 9.13 A, modes var% 12/11/7 (cum 30%)
  model 3:   203 frames ( 1.1%), medoid frame 33450, t = 66.902 ns, precision 1.56 A, RMSD to PDB 3GB1 model 1 4.28 A, modes var% 13/8/6 (cum 28%)
  model 4:   182 frames ( 1.0%), medoid frame 194380, t = 388.762 ns, precision 1.62 A, RMSD to PDB 3GB1 model 1 4.08 A, modes var% 15/10/8 (cum 33%)
  timings: modes 1.2s, reference 0.2s, write 0.1s


valid


## 3. What changes in the file

The categories are the same as in 1_basic. Three things differ:

- `_mdcif_mode_set.method` is `quasi-harmonic` for the states' mode sets (the global set stays
  `PCA`), and `atom_selection` records the selection.
- `_mdcif_mode_vector` has 434 rows per mode (one per atom) instead of 56.
- Quasi-harmonic eigenvalues are in amu·Å². The reader converts them to σ (Å), the standard
  deviation of the motion along the mode for all the atoms together; each atom moves σ times
  its own `(dx, dy, dz)`. Viewers draw arrows and movies from σ.

In [5]:
data = read_mdcif("gb1_heavy_mw.cif")
for set_id, row in data.mode_set_info.items():
    glob = row["scope"] == "global"
    ms = data.global_modes if glob else data.mode_sets[row["model_id"]]
    unit = "amu·Å²" if ms.mass_weighted else "Å²"
    lam = " ".join(f"{x:6.0f}" for x in ms.eigenvalues)
    sigma = " ".join(f"{x:4.1f}" for x in np.sqrt(ms.cartesian_variance))
    name = "global" if glob else f"model {row['model_id']}"
    print(f"set {set_id} ({name:7s}) {ms.method:14s} {len(ms.atom_indices)} atoms  "
          f"eigenvalues {lam} {unit:6s}  σ {sigma} Å")

set 1 (global ) PCA            434 atoms  eigenvalues   1687    364    248 Å²      σ 41.1 19.1 15.8 Å
set 2 (model 1) quasi-harmonic 434 atoms  eigenvalues   4289   2092   1128 amu·Å²  σ 18.0 12.7  9.3 Å
set 3 (model 2) quasi-harmonic 434 atoms  eigenvalues   1338   1150    781 amu·Å²  σ 10.0  9.3  7.6 Å
set 4 (model 3) quasi-harmonic 434 atoms  eigenvalues   1310    779    634 amu·Å²  σ  9.9  7.8  6.9 Å
set 5 (model 4) quasi-harmonic 434 atoms  eigenvalues   1597   1098    821 amu·Å²  σ 11.0  9.1  7.8 Å


## 4. Does the weighting matter here?

To check, compare each state's mass-weighted modes with plain PCA of the same frames. The
overlap of two modes is |v₁ · v₂|: 1 means they point the same way, 0 means they are unrelated.

In [6]:
from mdcif.modes import cluster_modes
from mdcif.trajectory import element_masses

masses = element_masses(traj.analysis)          # the standard masses save_mdcif used
for c, f in enumerate(clusters.representatives):
    frames, medoid = traj.raw[clusters.members(c)], traj.aligned[f]
    weighted = cluster_modes(frames, masses=masses, reference=medoid)
    plain = cluster_modes(frames, reference=medoid)
    overlap = np.abs((weighted.eigenvectors * plain.eigenvectors).sum(axis=(1, 2)))
    print(f"model {c + 1}: overlap of modes 1-3 = " + ", ".join(f"{x:.3f}" for x in overlap))

model 1: overlap of modes 1-3 = 1.000, 0.999, 1.000
model 2: overlap of modes 1-3 = 0.999, 0.998, 1.000
model 3: overlap of modes 1-3 = 1.000, 0.998, 0.997
model 4: overlap of modes 1-3 = 1.000, 0.998, 0.994


The overlaps are almost exactly 1, because C, N and O weigh about the same (12–16 amu). Mass
weighting matters when the masses differ a lot: when hydrogens (1 amu) are in the selection,
or with metal ions or ligands that hold heavier atoms such as S, P or halogens.

## 5. The same from the command line

From this folder, in a terminal, with the environment active (`conda activate mdcif`):

```bash
TOP=../../data/3gb1/meld/3gb1_HMR.prmtop
TRAJ=../../data/3gb1/meld/meld.nc.001

mdcif build $TOP $TRAJ --provenance provenance.yaml --start 25000 --step 10 \
    --selection "protein and element C N O" --mass-weighted --output gb1_heavy_mw.cif
mdcif info gb1_heavy_mw.cif                         # models, atoms and modes in the file
mdcif validate gb1_heavy_mw.cif                     # check against the PDBx/IHM/mdCIF dictionaries
mdcif landscape gb1_heavy_mw.cif --reference 3GB1   # PCs 1-3 plot: gb1_heavy_mw_landscape.png
```

| Part of `mdcif build` | Meaning |
|---|---|
| `$TOP $TRAJ` | the topology, then the trajectory |
| `--provenance provenance.yaml` | how the run was made; its `reference_PDB_id: 3GB1` makes the file an edited copy of 3GB1 |
| `--start 25000 --step 10` | from 50 ns on, every 10th frame, as above |
| `--selection "protein and element C N O"` | the atoms for every step (`selection=` above); keep the quotes |
| `--mass-weighted` | quasi-harmonic modes for each state (`mass_weighted=True` above); `--no-mass-weighted` for plain PCA |
| `--output gb1_heavy_mw.cif` | the file to write; a `.bcif` name gives BinaryCIF |

The `modes:` block of `provenance.yaml` already sets the selection and `mass_weighted: true`,
so `--selection` and `--mass-weighted` could be left out here. Options typed on the command
line win over the YAML. The build writes the same file as the Python cells above (and
replaces it). [3_cli](../3_cli/3_cli.ipynb) covers the other commands and options.